# Model Training

## Import necessary libraries

In [1]:
%pip install -qq -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Add current directory to Python path for imports
import os
import sys

# Add the parent directory (project root) to Python path so we can import from src
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
# Utility Functions
from src.utils import create_spark_session

# Create Spark session
spark, sedona = create_spark_session(app_name="DataCleansingSpark")

## Loading Datasets

In [4]:
from src.utils import read_config_path

# Load data using configuration file
filepath = read_config_path(key="raw_data_path")

df = spark.read.csv(
    filepath,
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    quote='"',
)

df.show(10)

+-----------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+-----------+--------+-------------+--------------------+---------+----+------------+--------------------+
|  ticket_id|               type|        organization|             comment|               photo|         photo_after|            coords|             address|subdistrict|district|     province|           timestamp|    state|star|count_reopen|       last_activity|
+-----------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+-----------+--------+-------------+--------------------+---------+----+------------+--------------------+
|2021-FYJTFP|        {ความสะอาด}|          เขตบางซื่อ|             ขยะเยอะ|https://storage.g...|                NULL|100.53084,13.81865|12/14 ถนน กรุงเทพ...|       NULL|    NULL|กรุงเทพมหานคร|2021-09-03 19:51:..

---

## Applying Cleansing Pipeline

In [5]:
from src.pipelines_spark import CleansingPipelineSpark

cleansing_pipeline = CleansingPipelineSpark(spark, sedona)
df_cleansed = cleansing_pipeline.transform(df)

df_cleansed.show(10)

+-----------+-------------------+--------------------+--------------------+--------------------+-----------+--------+-------------+--------------+---------------+--------------+------------------+-------------------+------------------+---------+--------+------+
|  ticket_id|               type|        organization|             comment|             address|subdistrict|district|     province|timestamp_date|timestamp_month|timestamp_year|last_activity_date|last_activity_month|last_activity_year|longitude|latitude|status|
+-----------+-------------------+--------------------+--------------------+--------------------+-----------+--------+-------------+--------------+---------------+--------------+------------------+-------------------+------------------+---------+--------+------+
|2021-CGPMUN|{น้ำท่วม,ร้องเรียน}|เขตประเวศ,ฝ่ายโยธ...|น้ำท่วมเวลาฝนตกแล...|189 เฉลิมพระเกียร...|    หนองบอน|  ประเวศ|กรุงเทพมหานคร|            19|              9|          2021|                21|                  

In [6]:
df_cleansed.printSchema()

root
 |-- ticket_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- organization: string (nullable = true)
 |-- comment: string (nullable = true)
 |-- address: string (nullable = true)
 |-- subdistrict: string (nullable = true)
 |-- district: string (nullable = true)
 |-- province: string (nullable = true)
 |-- timestamp_date: integer (nullable = true)
 |-- timestamp_month: integer (nullable = true)
 |-- timestamp_year: integer (nullable = true)
 |-- last_activity_date: integer (nullable = true)
 |-- last_activity_month: integer (nullable = true)
 |-- last_activity_year: integer (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- status: string (nullable = true)



---

## Applying Model Preparation Pipeline

In [7]:
from src.pipelines_spark import ModelPrepPipelineSpark

preparing_pipeline = ModelPrepPipelineSpark()
df_prepared = preparing_pipeline.transform(df_cleansed)

df_prepared.show(10)

+---------------+--------------+---------+--------+--------------------+--------------------+--------------------+---------------+
|timestamp_month|timestamp_year|longitude|latitude|     address_encoded|organization_encoded|        type_encoded|resolution_time|
+---------------+--------------+---------+--------+--------------------+--------------------+--------------------+---------------+
|              9|          2021|100.66709|13.67891|(2048,[834,1804],...|(1786,[10,53],[1....|(25,[7,8],[1.0,1.0])|            275|
|              9|          2021|100.52649| 13.7206|(2048,[348,426],[...|   (1786,[49],[1.0])|     (25,[14],[1.0])|            253|
|             12|          2021|100.59165| 13.8228|(2048,[802,1656],...|(1786,[31,108],[1...|(25,[0,8],[1.0,1.0])|            246|
|             12|          2021|100.59131| 13.8091|(2048,[802,1656],...|(1786,[31,172],[1...|      (25,[1],[1.0])|            456|
|             12|          2021|100.50848|13.77832|(2048,[1025,1114]...|   (1786,[5

In [8]:
df_prepared.printSchema()

root
 |-- timestamp_month: integer (nullable = true)
 |-- timestamp_year: integer (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- address_encoded: vector (nullable = true)
 |-- organization_encoded: vector (nullable = true)
 |-- type_encoded: vector (nullable = true)
 |-- resolution_time: integer (nullable = true)



## Saving Files

In [9]:
df_pandas = df_prepared.toPandas()
df_pandas.head()

,timestamp_month,timestamp_year,longitude,latitude,address_encoded,organization_encoded,type_encoded,resolution_time
0,9,2021,100.66709,13.67891,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, ...",275
1,9,2021,100.52649,13.72060,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",253
2,12,2021,100.59165,13.82280,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ...",246
3,12,2021,100.59131,13.80910,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",456
4,12,2021,100.50848,13.77832,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",516


In [10]:
import pandas as pd
from src.utils import get_data_dir

save_name = "model_training_spark.csv"
save_path = get_data_dir() / "processed" / save_name

pd.DataFrame.to_csv(
    df_pandas,
    save_path,
    index=False,
)

In [11]:
spark.stop()

---